# Notebook 01 — Data Extraction & Standardisation

**Purpose**: Parse every xlsx file in `obsidian_minerales_component_tables_from_articles/`,
standardise column names, source labels, and method tiers, then export one clean CSV per paper.

**Rules**:
- Raw xlsx files are NEVER modified
- Every decision is logged in `changelog.md` (update manually after running each section)
- See `reference_database/PRE_PHASE1_INVENTORY.md` for the extraction map

**Output files** go to `reference_database/sources/` (one CSV per geological source)
and a combined `reference_database/extracted_raw/` folder (one CSV per paper, pre-merge).

---

## Method Tiers (from audit_cross_reference.md Section E)
| Tier | Methods | Use |
|------|---------|-----|
| 1 | pXRF | Primary — pool directly |
| 2 | EDXRF (lab) | Pool with flag |
| 3 | LA-ICP-MS, ICP-AES/MS, PIXE, WDXRF | Reference-only |
| 4 | NAA/INAA, Electron Microprobe | Different element suite |

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path(r'c:\work\code\obsidian')
RAW_DIR    = ROOT / 'obsidian_minerales_component_tables_from_articles'
OUT_DIR    = ROOT / 'reference_database' / 'extracted_raw'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Paths ok. Raw dir:', RAW_DIR)
print('Files:', [f.name for f in RAW_DIR.glob('*.xlsx')])

In [ ]:
# ── Standard column names ──────────────────────────────────────────────────────
# All data will be normalised to these names.
# Anything not in this list gets prefixed with 'extra_'.

ELEMENT_COLS = [
    'Rb', 'Sr', 'Zr', 'Nb', 'Y', 'Fe', 'Mn', 'Ba', 'Zn', 'Ti',
    'Th', 'U', 'Pb', 'Ga', 'Al', 'Si', 'K', 'Ca', 'Li', 'B',
    'As', 'Cs', 'Co', 'Ni', 'Cu'
]
ERROR_COLS = [f'{e}_err' for e in ELEMENT_COLS]   # error/uncertainty columns
RANGE_COLS = [f'{e}_range_flag' for e in ELEMENT_COLS]  # original range strings

META_COLS = [
    'sample_id', 'source', 'site', 'paper', 'year',
    'method', 'method_tier', 'units', 'notes', 'verification_flag'
]

# ── Source label map ───────────────────────────────────────────────────────────
SOURCE_MAP = {
    # Göllü Dağ East
    'east gollu dag': 'EGD', 'gollu dag east': 'EGD', 'gld-e': 'EGD',
    'egd': 'EGD', 'göllü dağ-e': 'EGD', 'gollu dag-e': 'EGD',
    'kömürcü-kaletepe': 'EGD', 'komurcü': 'EGD', 'bozkoy': 'EGD',
    'kumurcu n': 'EGD', 'kumurcu sw': 'EGD', 'east gölüdag': 'EGD',
    'e. gollu dag': 'EGD', 'gollu dag (east)': 'EGD',
    # Göllü Dağ West
    'west gollu dag': 'WGD', 'gollu dag west': 'WGD', 'gld-w': 'WGD',
    'wgd': 'WGD', 'göllü dağ-w': 'WGD', 'gollu dag (west)': 'WGD',
    # Nenezi Dağ
    'nenezi dag': 'ND', 'nenezi dağ': 'ND', 'nnd': 'ND', 'nd': 'ND',
    'nenezi': 'ND',
    # Bingöl A
    'bingol a': 'BingolA', 'bingöl a': 'BingolA', 'bingol-a': 'BingolA',
    'bingöl-a': 'BingolA', 'bingöl_a': 'BingolA',
    # Bingöl B
    'bingol b': 'BingolB', 'bingöl b': 'BingolB', 'bingol-b': 'BingolB',
    'bingöl-b': 'BingolB', 'bingöl_b': 'BingolB',
    # Nemrut Dağ
    'nemrut dag': 'NemrutDag', 'nemrut dağ': 'NemrutDag',
    'nmrd': 'NemrutDag', 'nmrd1': 'NemrutDag', 'nmrd2': 'NemrutDag',
    'lake van': 'NemrutDag', 'nemrut dağ bingöl a': 'NemrutDag',
    # Others
    'meydan dag': 'MeydanDag', 'meydan dağ': 'MeydanDag', 'meydan_dag': 'MeydanDag',
    'mus': 'Mus', 'muş': 'Mus',
    'sevan': 'Sevan', 'znkt': 'Sevan',
    'urmia': 'Urmia', 'lake urmia': 'Urmia',
    'htmd': 'HTMD',  # Hazan/Hasan Dag — confirm from article
    'krud': 'KRUD',  # keep raw until confirmed
    'gld': 'EGD',    # Yellin papers use GLD = Göllü Dağ (assume EGD unless confirmed otherwise)
    'group 3d': 'Group3d',  # unconfirmed — investigate
}

def normalise_source(raw):
    """Map a raw source label to the standard label. Returns raw if unmapped."""
    if pd.isna(raw):
        return np.nan
    key = str(raw).strip().lower()
    return SOURCE_MAP.get(key, str(raw).strip())  # keep original if not in map

print('Source map loaded:', len(SOURCE_MAP), 'entries')

In [ ]:
# ── Helper: parse range strings ────────────────────────────────────────────────

def parse_range(val):
    """
    Detect and parse a range string like '166-194'.
    Returns (midpoint_float, original_string) or (float_val, None).

    PLAIN LANGUAGE: Some papers report a range of values for a source
    instead of a single mean. We convert that to the midpoint
    (e.g., 166-194 → 180) so we can use it numerically.
    The original string is saved in a _range_flag column so nothing is lost.
    """
    if pd.isna(val):
        return np.nan, None
    s = str(val).strip()
    if '-' in s and not s.startswith('-'):
        parts = s.split('-')
        try:
            lo, hi = float(parts[0]), float(parts[1])
            return (lo + hi) / 2, s
        except ValueError:
            pass
    try:
        return float(s), None
    except ValueError:
        return np.nan, s  # unparseable — flag it

# Test
assert parse_range('166-194') == (180.0, '166-194')
assert parse_range('185') == (185.0, None)
assert parse_range(np.nan) == (np.nan, None)
print('parse_range OK')

In [ ]:
# ── Helper: rename XRF line notation to element symbols ───────────────────────
# e.g.  RbKa1 → Rb,  SrKa1 → Sr,  ZrKa1 → Zr

XRF_LINE_MAP = {
    'mnka1': 'Mn', 'feka1': 'Fe', 'znka1': 'Zn', 'gaka1': 'Ga',
    'thla1': 'Th', 'rbka1': 'Rb', 'srka1': 'Sr', 'yka1': 'Y',
    'zrka1': 'Zr', 'nbka1': 'Nb', 'tika1': 'Ti', 'pbla1': 'Pb',
    'bala1': 'Ba', 'uka1': 'U',
}

def clean_colname(col):
    """Strip whitespace, map XRF line notation to symbol, else title-case."""
    c = str(col).strip()
    return XRF_LINE_MAP.get(c.lower(), c)

print('XRF line map loaded:', len(XRF_LINE_MAP), 'entries')

In [ ]:
# ── Helper: add standard metadata columns ─────────────────────────────────────

def add_meta(df, paper, year, method, method_tier, units='ppm'):
    """Attach consistent metadata columns to a DataFrame."""
    df = df.copy()
    for col in ['sample_id', 'source', 'site', 'notes', 'verification_flag']:
        if col not in df.columns:
            df[col] = np.nan
    df['paper']       = paper
    df['year']        = year
    df['method']      = method
    df['method_tier'] = method_tier
    df['units']       = units
    df['verification_flag'] = False
    return df

print('Helpers ready.')

---
## TIER 1 — pXRF Papers
### 1A. Schechter et al. 2016

**What this paper is**: pXRF measurements of obsidian from South Levant sites.  
**File**: `Schechter et al 2016.xlsx` → Sheet `גיליון1`  
**Columns**: ID#, MnKa1, FeKa1, ZnKa1, GaKa1, ThLa1, RbKa1, SrKa1, YKa1, ZrKa1, NbKa1, Source  
**Action**: Rename XRF-line headers → element symbols. Normalise source labels.

In [ ]:
# ── 1A. Schechter et al. 2016 ─────────────────────────────────────────────────

df_schechter = pd.read_excel(
    RAW_DIR / 'Schechter et al 2016.xlsx',
    sheet_name=0,   # only one sheet
    dtype=str       # read everything as string first to catch range values
)

# Rename columns
df_schechter.columns = [clean_colname(c) for c in df_schechter.columns]
df_schechter = df_schechter.rename(columns={'ID #': 'sample_id'})

# Convert numeric element cols
element_present = [c for c in df_schechter.columns if c in ELEMENT_COLS]
for col in element_present:
    df_schechter[col] = pd.to_numeric(df_schechter[col], errors='coerce')

# Normalise source
df_schechter['source'] = df_schechter['Source'].apply(normalise_source)
df_schechter = df_schechter.drop(columns=['Source'], errors='ignore')

# Add metadata
df_schechter = add_meta(df_schechter,
    paper='Schechter et al. 2016', year=2016, method='pXRF', method_tier=1)

print(f'Schechter 2016: {len(df_schechter)} rows, {len(element_present)} element cols')
print('Sources:', df_schechter['source'].value_counts().to_dict())
df_schechter[['sample_id', 'source'] + element_present[:6]].head(3)

In [ ]:
# Spot-check Schechter et al. 2016 against article (Schechter_et_al_2016.txt)
# Article Table (line 690): EZPR 10585.1: Mn=447, Fe=5127, Zn=30, Ga=21, Th=15, Rb=126
article_vals = {'Mn': 447, 'Fe': 5127, 'Zn': 30, 'Ga': 21, 'Th': 15, 'Rb': 126}
row = df_schechter[df_schechter['sample_id'] == 'EZPR 10585.1'].iloc[0]
for el, expected in article_vals.items():
    actual = row[el]
    status = 'PASS' if abs(actual - expected) < 0.5 else f'FAIL got {actual}'
    print(f'  {el:3s}: expected={expected}  {status}')
print()
print('NOTE: Schechter 2016 labels Gollu Dag without E/W distinction -> mapped to GolluDag')
print('VERIFICATION: 2026-03-25 | EZPR 10585.1 spot-check | log in changelog')

In [ ]:
# Save
out_path = OUT_DIR / 'schechter_2016.csv'
df_schechter.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

---
### 1B. Frahm 2013 — HHpXRF validity, Göllü Dağ debitage

**What this paper is**: Test of handheld pXRF on irregular obsidian chip debris.  
94–100% source assignment success rate v. EMPA ground truth.  
**Article**: `Frahm_2013.txt` ✅  
**File**: `Frahm 2013.xlsx` → Sheet `גיליון1`  
**Columns**: Point, Zr (±err), Sr (±err), Rb (±err), Zn (±err), Mn (±err), Ti  
**Note**: These are measurements of *archaeological debitage* (from Göllü Dağ source).  
All 67 samples assigned to Göllü Dağ — they serve as site data, not geological source reference.

In [ ]:
# ── 1B. Frahm 2013 ────────────────────────────────────────────────────────────

df_frahm13 = pd.read_excel(
    RAW_DIR / 'Frahm 2013.xlsx',
    sheet_name=0,
    dtype=str
)

# The columns are: Point, Zr, Error, Sr, Error, Rb, Error, Zn, Error, Mn, Error, Ti
# Rename: paired value+error columns
raw_cols = list(df_frahm13.columns)
print('Raw columns:', raw_cols)

# Build rename map: first 'Error' after Zr → Zr_err, etc.
elements_order = ['Zr', 'Sr', 'Rb', 'Zn', 'Mn']
new_cols = ['sample_id']
ei = 0
for c in raw_cols[1:]:  # skip 'Point'
    c_clean = clean_colname(c)
    if c_clean in ELEMENT_COLS:
        new_cols.append(c_clean)
        ei += 1
    elif c_clean.lower() in ('error', 'err', 'error '):
        # assign error to the previous element
        prev_el = new_cols[-1] if new_cols[-1] in ELEMENT_COLS else None
        new_cols.append(f'{prev_el}_err' if prev_el else 'Unknown_err')
    else:
        new_cols.append(c_clean)

df_frahm13.columns = new_cols
print('Renamed columns:', list(df_frahm13.columns))

# Convert numeric
for col in [c for c in df_frahm13.columns if c != 'sample_id']:
    df_frahm13[col] = pd.to_numeric(df_frahm13[col], errors='coerce')

# All samples are from Göllü Dağ (as per paper) — likely EGD (confirm in article)
df_frahm13['source'] = 'EGD'  # ← confirm from article Table/text
df_frahm13['site']   = 'Tell Mozan'  # site in Syria where artefacts were from

df_frahm13 = add_meta(df_frahm13,
    paper='Frahm 2013', year=2013, method='pXRF (HHpXRF)', method_tier=1)

print(f'Frahm 2013: {len(df_frahm13)} rows')
df_frahm13[['sample_id', 'source', 'Rb', 'Rb_err', 'Sr', 'Zr']].head(3)

In [ ]:
# Spot-check Frahm 2013 against article text (Frahm_2013.txt)
# The paper reports descriptive stats; check a few sample values match table in article.
# Log results in changelog.md
print('>> Verify source assignment: are all 67 points assigned to EGD in article?')
print('>> Check article for split between EGD sub-outcrops (Kömürcü vs Kaletepe)')

# Save
out_path = OUT_DIR / 'frahm_2013.csv'
df_frahm13.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

---
### 1C. Milic 2014 — pXRF source summary (range values)

**What this paper is**: pXRF of obsidian from Neolithic Balkans/Anatolia.  
**File**: `Milic 2014.xlsx` → Sheet `גיליון1` (15 rows, 3 elements)  
**Also in**: `data1.xlsx` → Sheet `Milic 2014` (41 rows, side-by-side layout)  
**Columns**: Source, Rb (ppm), Sr (ppm), Zr (ppm)  
**Special**: Range strings like '166-194' → parse to midpoint, preserve original.

In [ ]:
# ── 1C. Milic 2014 (standalone file) ─────────────────────────────────────────

df_milic_raw = pd.read_excel(
    RAW_DIR / 'Milic 2014.xlsx',
    sheet_name=0,
    dtype=str
)
print('Milic 2014 raw:\n', df_milic_raw.to_string())

In [ ]:
# Rename columns
df_milic_raw.columns = [clean_colname(c) for c in df_milic_raw.columns]
# Expected: Source, Rb (ppm), Sr (ppm), Zr (ppm)  →  source_raw, Rb, Sr, Zr
col_rename = {}
for c in df_milic_raw.columns:
    base = c.replace(' (ppm)', '').replace(' (ppm) ', '').strip()
    col_rename[c] = base
df_milic_raw = df_milic_raw.rename(columns=col_rename)
print('Cleaned columns:', list(df_milic_raw.columns))

# Parse range strings
"""
PLAIN LANGUAGE:
Milic 2014 reports ranges for each source (e.g., Göllü Dağ Rb = '166-194').
These are the min-max observed values across all samples from that source.
We convert them to a midpoint for numeric analysis but store the original range.
"""
df_milic = pd.DataFrame()
df_milic['source_raw'] = df_milic_raw.get('Source', df_milic_raw.iloc[:, 0])
df_milic['source'] = df_milic['source_raw'].apply(normalise_source)

for el in ['Rb', 'Sr', 'Zr']:
    if el in df_milic_raw.columns:
        parsed = df_milic_raw[el].apply(parse_range)
        df_milic[el]               = [v[0] for v in parsed]
        df_milic[f'{el}_range_flag'] = [v[1] for v in parsed]

df_milic = add_meta(df_milic,
    paper='Milic 2014', year=2014, method='pXRF', method_tier=1)

print(f'Milic 2014: {len(df_milic)} rows')
df_milic[['source', 'Rb', 'Rb_range_flag', 'Sr', 'Zr']].head()

In [ ]:
# Spot-check pending — no article txt yet
print('⚠️  Spot-check pending — article text for Milic 2014 not yet available')

out_path = OUT_DIR / 'milic_2014.csv'
df_milic.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

---
### 1D. Campbell & Healey 2016 — largest Tier 1 source dataset

**What this paper is**: pXRF of geological source samples from multiple Anatolian sources.  
**File**: `campbell and healey 1.xlsx` — 8 sheets, ~880 total source rows  
**Elements**: Al, Si, K, Ca, Ti, Mn, Fe, Zn, Rb, Sr, Y, Zr + more  
**Sheets**: Kenan sources (site data), Nemrut Dağ Bingöl A, Nemrut Dağ (n=340), Muş (n=3), Meydan Dağ, Group 3d (n=279 — investigate), Bingöl A (n=123), Bingöl B (n=123)

In [ ]:
# ── 1D. Campbell & Healey 2016 ───────────────────────────────────────────────

ch_file = RAW_DIR / 'campbell and healey 1.xlsx'
xl_ch = pd.ExcelFile(ch_file)
print('Sheets:', xl_ch.sheet_names)

In [ ]:
# Source sheets to extract (excludes site/archaeological data sheet)
SOURCE_SHEETS_CH = [
    'Nemrut Dağ Bingöl A',
    'Nemrut Dağ',
    'Muş',
    'Meydan Dağ',
    'Group 3d',
    'Bingöl A',
    'Bingöl B',
]

# SITE sheet (archaeological, not geological source)
SITE_SHEET_CH = 'Kenan sources assignments v3'

frames_ch = []
for sname in SOURCE_SHEETS_CH:
    df_s = pd.read_excel(ch_file, sheet_name=sname, dtype=str)
    df_s.columns = [clean_colname(c) for c in df_s.columns]

    # Convert numeric element cols
    for col in [c for c in df_s.columns if c in ELEMENT_COLS]:
        df_s[col] = pd.to_numeric(df_s[col], errors='coerce')

    # Source: use the sheet name as source, normalised
    if 'Source' in df_s.columns:
        df_s['source'] = df_s['Source'].apply(normalise_source)
    else:
        df_s['source'] = normalise_source(sname)

    # Sample ID
    id_col = next((c for c in df_s.columns if 'reference' in c.lower() or 'sample' in c.lower()), None)
    df_s['sample_id'] = df_s[id_col] if id_col else pd.RangeIndex(len(df_s)).astype(str)

    df_s = add_meta(df_s, paper='Campbell & Healey 2016', year=2016,
                    method='pXRF', method_tier=1)
    df_s['site'] = 'geological source'

    frames_ch.append(df_s)
    print(f'  {sname:35s}: {len(df_s):4d} rows | source → {df_s["source"].unique()}')

df_ch = pd.concat(frames_ch, ignore_index=True)
print(f'\nCampbell & Healey 2016 total: {len(df_ch)} rows')
print('Sources:', df_ch['source'].value_counts().to_dict())

In [ ]:
# ⚠️  INVESTIGATION NEEDED: Group 3d
"""
PLAIN LANGUAGE:
One of the sheets is called 'Group 3d' and has 279 rows.
We don't know which obsidian source this corresponds to.
It could be a sub-group of an existing source, or an unnamed/new source.
Action: check article text when available. For now, labelled 'Group3d' to avoid wrong assignments.
"""
print('Group3d rows:', len(df_ch[df_ch['source'] == 'Group3d']))
print('⚠️  Group3d identity unknown — needs article confirmation before use in reference database')

In [ ]:
# Spot-check pending — no article txt yet
print('⚠️  Spot-check pending — article text for Campbell & Healey 2016 not yet available')

out_path = OUT_DIR / 'campbell_healey_2016.csv'
df_ch.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

---
### 1E. Morgan 2015 — 714 individual pXRF measurements

**File**: `Morgan 2015.xlsx` — Three relevant sheets  
- `Table1`: 714 rows of element measurements (Analysis ID + 15 elements)  
- `Table2`: 714 rows of metadata (Analysis ID, Site, Source)  
- `Dhemenegaki` + `Sta Nychia`: Aegean sources (Greek islands) — extract but tag separately  
**Join Table1+Table2 on `Analysis ID`**

In [ ]:
# ── 1E. Morgan 2015 ───────────────────────────────────────────────────────────

mfile = RAW_DIR / 'Morgan 2015.xlsx'

df_m1 = pd.read_excel(mfile, sheet_name='Table1', dtype=str)
df_m2 = pd.read_excel(mfile, sheet_name='Table2', dtype=str)

# Clean column names
df_m1.columns = [clean_colname(c) for c in df_m1.columns]
df_m2.columns = [clean_colname(c) for c in df_m2.columns]

print('Table1 cols:', list(df_m1.columns))
print('Table2 cols:', list(df_m2.columns))

In [ ]:
# Identify the join key (Analysis ID)
id_col1 = next(c for c in df_m1.columns if 'analysis' in c.lower() or 'id' in c.lower())
id_col2 = next(c for c in df_m2.columns if 'analysis' in c.lower() or 'id' in c.lower())
print(f'Join on: Table1["{id_col1}"] ↔ Table2["{id_col2}"]')

# Join
df_morgan = df_m1.merge(df_m2, left_on=id_col1, right_on=id_col2, how='left', suffixes=('', '_meta'))
df_morgan = df_morgan.rename(columns={id_col1: 'sample_id'})

# Convert element cols numeric
el_morgan = [c for c in df_morgan.columns if c in ELEMENT_COLS]
for col in el_morgan:
    df_morgan[col] = pd.to_numeric(df_morgan[col], errors='coerce')

# Source
src_col = next((c for c in df_morgan.columns if c.lower() == 'source'), None)
df_morgan['source'] = df_morgan[src_col].apply(normalise_source) if src_col else np.nan

# Site
site_col = next((c for c in df_morgan.columns if 'site' in c.lower()), None)
df_morgan['site'] = df_morgan[site_col] if site_col else np.nan

df_morgan = add_meta(df_morgan, paper='Morgan 2015', year=2015, method='pXRF', method_tier=1)
df_morgan['region'] = 'Near East'  # default; Dhemenegaki/Sta Nychia handled separately

print(f'Morgan 2015 (joined): {len(df_morgan)} rows, {len(el_morgan)} element cols')
print('Sources:', df_morgan['source'].value_counts().head(10).to_dict())

out_path = OUT_DIR / 'morgan_2015.csv'
df_morgan.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')
print('⚠️  Spot-check pending — article text for Morgan 2015 not yet available')

In [ ]:
# Aegean source sheets — extract separately, tagged
aegean_frames = []
for sname in ['Dhemenegaki', 'Sta Nychia']:
    try:
        df_ae = pd.read_excel(mfile, sheet_name=sname, dtype=str)
        df_ae.columns = [clean_colname(c) for c in df_ae.columns]
        for col in [c for c in df_ae.columns if c in ELEMENT_COLS]:
            df_ae[col] = pd.to_numeric(df_ae[col], errors='coerce')
        df_ae = add_meta(df_ae, paper='Morgan 2015', year=2015, method='pXRF', method_tier=1)
        df_ae['source'] = sname
        df_ae['region'] = 'Aegean'
        aegean_frames.append(df_ae)
        print(f'  {sname}: {len(df_ae)} rows')
    except Exception as e:
        print(f'  {sname}: ERROR - {e}')

if aegean_frames:
    df_aegean = pd.concat(aegean_frames, ignore_index=True)
    out_path = OUT_DIR / 'morgan_2015_aegean.csv'
    df_aegean.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'Saved Aegean subset: {out_path}')

---
## TIER 2 — EDXRF Papers
### 2A. Carter & Shackley 2007

**Article**: `Carter_and_Shackley_2007.txt` ✅  
**File**: `data2.xlsx` → Sheet `Carter and Shackley 2007`  
**Columns**: Sample, Ti, Mn, Fe, Zn, Ga, Th, Rb, Sr, Y, Zr, Nb, Nb/Zr, Y/Zr, Source  
**56 rows** — mix of geological source samples and archaeological samples.

In [ ]:
# ── 2A. Carter & Shackley 2007 ────────────────────────────────────────────────

d2_file = RAW_DIR / 'data2.xlsx'

df_cs07 = pd.read_excel(d2_file, sheet_name='Carter and Shackley 2007', dtype=str)
df_cs07.columns = [clean_colname(c) for c in df_cs07.columns]
df_cs07 = df_cs07.rename(columns={'Sample': 'sample_id'})

el_cs07 = [c for c in df_cs07.columns if c in ELEMENT_COLS]
for col in el_cs07:
    df_cs07[col] = pd.to_numeric(df_cs07[col], errors='coerce')

# Preserve ratio columns as extra info
for ratio_col in ['Nb/Zr', 'Y/Zr']:
    if ratio_col in df_cs07.columns:
        df_cs07[f'extra_{ratio_col.replace("/","_")}'] = pd.to_numeric(df_cs07[ratio_col], errors='coerce')
        df_cs07 = df_cs07.drop(columns=[ratio_col])

df_cs07['source'] = df_cs07['Source'].apply(normalise_source)
df_cs07 = df_cs07.drop(columns=['Source'], errors='ignore')
df_cs07 = add_meta(df_cs07, paper='Carter & Shackley 2007', year=2007,
                   method='EDXRF', method_tier=2)

print(f'Carter & Shackley 2007: {len(df_cs07)} rows, {len(el_cs07)} elements')
print('Sources:', df_cs07['source'].value_counts().to_dict())
df_cs07[['sample_id', 'source', 'Rb', 'Sr', 'Zr', 'Nb']].head()

In [ ]:
# TODO: Spot-check against Carter_and_Shackley_2007.txt
# The article IS available — do this now.
# Read the article, find Table with source data, compare 3-5 rows.
print('>>> ACTION: Open articles/Carter_and_Shackley_2007.txt and spot-check 3 rows')
print('>>> Log results in changelog.md as VERIFICATION entry')

out_path = OUT_DIR / 'carter_shackley_2007.csv'
df_cs07.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

---
### 2B. Carter et al. 2017

**Article**: `Carter_2017_Investigating_Pottery_Neolithic_Shaar_Hagolan.txt` ✅  
**File**: `data2.xlsx` → Sheet `Carter et al. 2017`  
**Columns**: Sample, Source, Ti, Mn, Fe, Rb, Sr, Y, Zr, Nb, Ba, Pb, Th — 35 rows

In [ ]:
# ── 2B. Carter et al. 2017 ───────────────────────────────────────────────────

df_ca17 = pd.read_excel(d2_file, sheet_name='Carter et al. 2017', dtype=str)
df_ca17.columns = [clean_colname(c) for c in df_ca17.columns]
df_ca17 = df_ca17.rename(columns={'Sample': 'sample_id'})

el_ca17 = [c for c in df_ca17.columns if c in ELEMENT_COLS]
for col in el_ca17:
    df_ca17[col] = pd.to_numeric(df_ca17[col], errors='coerce')

df_ca17['source'] = df_ca17['Source'].apply(normalise_source)
df_ca17 = df_ca17.drop(columns=['Source'], errors='ignore')
df_ca17 = add_meta(df_ca17, paper='Carter et al. 2017', year=2017,
                   method='EDXRF', method_tier=2)
df_ca17['site'] = "Sha'ar Hagolan"  # Pottery Neolithic site

print(f'Carter et al. 2017: {len(df_ca17)} rows')
print('Sources:', df_ca17['source'].value_counts().to_dict())

out_path = OUT_DIR / 'carter_2017.csv'
df_ca17.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

---
### 2C. Carter et al. 2013 (Kenan Tepe)

**Article**: Not yet available (distinct from Tell Aswad and Körtik Tepe Carter 2013 papers)  
**File**: `data2.xlsx` → Sheet `Carter et al 2013`  
**Columns**: (blank first col), Ti, Mn, Fe, Zn, Ga, Rb, Sr, Y, Zr, Nb, Ba, Pb, Th, Source  
**121 rows** — Kenan Tepe site data, Bingöl source assignments

In [ ]:
# ── 2C. Carter et al. 2013 (Kenan Tepe) ──────────────────────────────────────

df_ca13 = pd.read_excel(d2_file, sheet_name='Carter et al 2013', dtype=str)
raw_cols_ca13 = list(df_ca13.columns)
df_ca13.columns = [clean_colname(c) for c in df_ca13.columns]

# First column is blank in header but likely sample ID
if df_ca13.columns[0] in ('', 'nan', 'None'):
    df_ca13 = df_ca13.rename(columns={df_ca13.columns[0]: 'sample_id'})

el_ca13 = [c for c in df_ca13.columns if c in ELEMENT_COLS]
for col in el_ca13:
    df_ca13[col] = pd.to_numeric(df_ca13[col], errors='coerce')

src_col = next((c for c in df_ca13.columns if 'source' in c.lower()), None)
df_ca13['source'] = df_ca13[src_col].apply(normalise_source) if src_col else np.nan
df_ca13 = add_meta(df_ca13, paper='Carter et al. 2013 (Kenan Tepe)', year=2013,
                   method='EDXRF', method_tier=2)
df_ca13['site'] = 'Kenan Tepe'

print(f'Carter et al. 2013 (Kenan Tepe): {len(df_ca13)} rows')
print('Sources:', df_ca13['source'].value_counts().to_dict())

out_path = OUT_DIR / 'carter_2013_kenan_tepe.csv'
df_ca13.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')
print('⚠️  Spot-check pending — article text not yet available')

---
### 2D. Rosenberg & Carter et al. 2022 (Tel Tsaf)

**Article**: `Carter_2022_Obsidian_Beads_Tel_Tsaf.txt` ✅  
**File**: `Carter_Rosenberg_2022_Tel_Tsaf.xlsx`  
- `Table5_Bead_Measurements`: 24 rows — bead measurements (archaeological)  
- `Table6_Source_Samples`: 44 rows — geological source reference  
**Elements**: Mn, Fe, Zn, Ga, Rb, Sr, Y, Zr, Nb, Th

In [ ]:
# ── 2D. Rosenberg & Carter 2022 ──────────────────────────────────────────────

tsaf_file = RAW_DIR / 'Carter_Rosenberg_2022_Tel_Tsaf.xlsx'

# Table 6: geological source reference data
df_tsaf_src = pd.read_excel(tsaf_file, sheet_name='Table6_Source_Samples', dtype=str)
df_tsaf_src.columns = [clean_colname(c) for c in df_tsaf_src.columns]

src_col = next((c for c in df_tsaf_src.columns if 'source' in c.lower()), None)
df_tsaf_src['source'] = df_tsaf_src[src_col].apply(normalise_source) if src_col else np.nan

el_tsaf = [c for c in df_tsaf_src.columns if c in ELEMENT_COLS]
for col in el_tsaf:
    df_tsaf_src[col] = pd.to_numeric(df_tsaf_src[col], errors='coerce')

df_tsaf_src = add_meta(df_tsaf_src, paper='Rosenberg & Carter et al. 2022', year=2022,
                       method='EDXRF', method_tier=2)
df_tsaf_src['site'] = 'geological source'

print(f'Tel Tsaf Table6 (source): {len(df_tsaf_src)} rows')
print('Sources:', df_tsaf_src['source'].value_counts().to_dict())

# Table 5: archaeological bead measurements
df_tsaf_beads = pd.read_excel(tsaf_file, sheet_name='Table5_Bead_Measurements', dtype=str)
df_tsaf_beads.columns = [clean_colname(c) for c in df_tsaf_beads.columns]

# Table 5 has a 'Stat' column (mean/SD rows) — check
print('\nTable5 columns:', list(df_tsaf_beads.columns))
print('Stat values:', df_tsaf_beads.get('Stat', pd.Series()).unique())

out_tsaf_src = OUT_DIR / 'rosenberg_carter_2022_sources.csv'
df_tsaf_src.to_csv(out_tsaf_src, index=False, encoding='utf-8-sig')
out_tsaf_beads = OUT_DIR / 'rosenberg_carter_2022_tel_tsaf_beads.csv'
df_tsaf_beads.to_csv(out_tsaf_beads, index=False, encoding='utf-8-sig')
print(f'\nSaved source data:  {out_tsaf_src}')
print(f'Saved bead data:    {out_tsaf_beads}')

---
## TIER 3 — LA-ICP-MS / PIXE / WDXRF
### 3A. Khalidi, Gratuze & Boucetta 2009

**Article**: `khalidi Gratuze 2009.txt` ✅  
**File**: `Khalidi Gratuze Boucetta 2009.xlsx` — 2 sheets  
- Sheet2 (גיליון2): 4 rows, 24 elements — source summary data (LA-ICP-MS)  
- Sheet1 (גיליון1): 23 rows, oxide% format for Bingöl A/B, Meydan Dağ

In [ ]:
# ── 3A. Khalidi et al. 2009 ───────────────────────────────────────────────────

kgb_file = RAW_DIR / 'Khalidi Gratuze Boucetta 2009.xlsx'
xl_kgb = pd.ExcelFile(kgb_file)
print('Sheets:', xl_kgb.sheet_names)

# Sheet 2 = main elemental data
df_kgb2 = pd.read_excel(kgb_file, sheet_name=1, dtype=str)  # index 1
print('\nSheet2 shape:', df_kgb2.shape)
print(df_kgb2.to_string())

In [ ]:
"""
PLAIN LANGUAGE:
This sheet is transposed — element names are in the first column ('Oxide'),
and source names are in subsequent columns.
We need to rotate it so each source is a row and each element is a column.
"""

# The 'Oxide' column contains element/oxide names; other cols = sources
# Set 'Oxide' as index, then transpose
df_kgb2 = df_kgb2.set_index(df_kgb2.columns[0]).T.reset_index()
df_kgb2.columns.name = None
df_kgb2 = df_kgb2.rename(columns={'index': 'source_raw'})

print('After transpose:')
print(df_kgb2.to_string())

In [ ]:
df_kgb2['source'] = df_kgb2['source_raw'].apply(normalise_source)

# Convert numeric
for col in [c for c in df_kgb2.columns if c not in ('source_raw', 'source')]:
    df_kgb2[col] = pd.to_numeric(df_kgb2[col], errors='coerce')

df_kgb2 = add_meta(df_kgb2, paper='Khalidi, Gratuze & Boucetta 2009', year=2009,
                   method='LA-ICP-MS', method_tier=3)

out_path = OUT_DIR / 'khalidi_gratuze_2009.csv'
df_kgb2.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')
print('Note: Only 4 rows — these are source means, not individual measurements')

---
### 3B. Frahm & Hauck 2017 — multi-method cross-comparison

**File**: `Frahm and Hauck 2017.xlsx`  
- `MAIN`: 252 rows — mixed methods (pXRF, EDXRF, NAA, LA-ICP-MS, WDXRF)  
- `Frahm and Hauck 2017-Gollu Dag`: 17 rows — cross-method comparison for Göllü Dağ  
**Critical**: tag each row with its method from the data — mixed tiers in one sheet.

In [ ]:
# ── 3B. Frahm & Hauck 2017 ────────────────────────────────────────────────────

fh_file = RAW_DIR / 'Frahm and Hauck 2017.xlsx'

df_fh_main = pd.read_excel(fh_file, sheet_name='MAIN', dtype=str)
df_fh_main.columns = [clean_colname(c) for c in df_fh_main.columns]
print('MAIN cols:', list(df_fh_main.columns))
print(df_fh_main.head())

In [ ]:
# Columns: Source, Specimen, Nb (ppm), Zr (ppm), Sr (ppm), Rb (ppm), Fe (ppm)
# Need to identify method — check if 'Specimen' or a method column exists

# Strip unit notation from column names
df_fh_main.columns = [c.replace(' (ppm)', '').strip() for c in df_fh_main.columns]

df_fh_main['source'] = df_fh_main['Source'].apply(normalise_source)
df_fh_main = df_fh_main.rename(columns={'Specimen': 'sample_id'})

el_fh = [c for c in df_fh_main.columns if c in ELEMENT_COLS]
for col in el_fh:
    df_fh_main[col] = pd.to_numeric(df_fh_main[col], errors='coerce')

# Method per row: inspect unique values of Source or Specimen to find method info
print('Source values:', df_fh_main['source'].unique()[:15])

# If no method column exists, assign tier=3 for now (mixed — review article)
df_fh_main = add_meta(df_fh_main, paper='Frahm & Hauck 2017', year=2017,
                      method='mixed (see paper)', method_tier=3)

print(f'Frahm & Hauck 2017 MAIN: {len(df_fh_main)} rows')
print('⚠️  Method tier per row needs confirmation from article — currently all assigned tier=3')

out_path = OUT_DIR / 'frahm_hauck_2017_main.csv'
df_fh_main.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}')

In [ ]:
# Gollu Dag cross-method comparison sheet
df_fh_gd = pd.read_excel(fh_file, sheet_name='Frahm and Hauck 2017-Gollu Dag', dtype=str)
print('Gollu Dag sheet:')
print(df_fh_gd.to_string())

In [ ]:
"""
PLAIN LANGUAGE — this sheet:
Each row is a different published study (Reference) using a different technique.
The columns show what each technique measured for Göllü Dağ source.
Paired columns (e.g., Nb + blank) likely = mean + SD.
This is the most important cross-method comparison table in your dataset.
It lets us see how much the same source looks different depending on the method.
"""
print('>>> ACTION: Read article (Frahm_Hauck_2017 — article txt not yet available)')
print('>>> Confirm which paired column is mean vs SD')
print('>>> Then update method_tier per row to match the technique')

out_path = OUT_DIR / 'frahm_hauck_2017_gollu_dag_crossmethod.csv'
df_fh_gd.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved (raw, needs cleanup): {out_path}')

---
## TIER 4 — NAA / Electron Microprobe (reference-only)
### 4A. Yellin & Perlman 1980 + 1981 and Yellin et al. 1996

**Articles**: `Yellin_and_Perlman_1980.txt` ✅, `Yellin_and_Perlman_1981.txt` ✅  
**Important**: NAA measures rare earth elements (La, Ce, Sm, Eu, Yb) and INAA-specific elements — NOT the same as pXRF (Rb, Sr, Zr, Nb). **Cannot directly compare** to Tier 1/2. Use only as source assignment reference.  
**Structure**: Transposed — elements as rows, sources as columns.

In [ ]:
# ── 4A. Yellin & Perlman 1980/1981 and Yellin et al. 1996 ────────────────────

for sheet_name, paper, year, out_name in [
    ('Yellin and Perlman 1980', 'Yellin & Perlman 1980', 1980, 'yellin_perlman_1980'),
    ('Yellin et al 1996',       'Yellin et al. 1996',    1996, 'yellin_1996'),
    ('Yellin and Perlman 1981', 'Yellin & Perlman 1981', 1981, 'yellin_perlman_1981'),
]:
    df_y = pd.read_excel(d2_file, sheet_name=sheet_name, dtype=str, header=None)
    print(f'\n{sheet_name} raw shape: {df_y.shape}')
    print(df_y.to_string())

In [ ]:
"""
PLAIN LANGUAGE — NAA/INAA data:
These papers from the 1980s used nuclear activation analysis — a very precise
method that bombards samples with neutrons. It measures different elements
than pXRF (rare earth elements, Sc, Cs, La, etc.).
There is almost no overlap with pXRF elements except Fe.
We keep these for historical reference and to confirm which source Neolithic
Levantine sites were supplied from — but we cannot plot them on the same biplots.
"""

# Process each — they're small (8-18 rows) and transposed
# The first column = element names; subsequent columns = sources
# We transpose and keep as-is for reference

for sheet_name, paper, year, out_name in [
    ('Yellin and Perlman 1980', 'Yellin & Perlman 1980', 1980, 'yellin_perlman_1980'),
    ('Yellin et al 1996',       'Yellin et al. 1996',    1996, 'yellin_1996'),
    ('Yellin and Perlman 1981', 'Yellin & Perlman 1981', 1981, 'yellin_perlman_1981'),
]:
    df_y = pd.read_excel(d2_file, sheet_name=sheet_name, dtype=str, header=None)

    # First non-empty row determines headers; first col = element names
    # Find first row with data
    df_y = df_y.dropna(how='all').dropna(axis=1, how='all')
    headers = df_y.iloc[0].fillna('').tolist()
    df_y = df_y.iloc[1:].copy()
    df_y.columns = headers
    df_y = df_y.rename(columns={headers[0]: 'element'})

    # Transpose so sources = rows, elements = columns
    df_y_T = df_y.set_index('element').T.reset_index()
    df_y_T.columns.name = None
    df_y_T = df_y_T.rename(columns={'index': 'source_raw'})
    df_y_T['source'] = df_y_T['source_raw'].apply(normalise_source)

    # Convert numeric
    for col in [c for c in df_y_T.columns if c not in ('source_raw', 'source')]:
        df_y_T[col] = pd.to_numeric(df_y_T[col], errors='coerce')

    df_y_T = add_meta(df_y_T, paper=paper, year=year,
                      method='NAA/INAA', method_tier=4)

    out_path = OUT_DIR / f'{out_name}.csv'
    df_y_T.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'{paper}: {len(df_y_T)} source rows → {out_path.name}')

---
## Summary & Next Steps

In [ ]:
import glob

print('=== Extraction Summary ===')
csv_files = sorted(Path(OUT_DIR).glob('*.csv'))
total_rows = 0
for f in csv_files:
    df = pd.read_csv(f)
    print(f'  {f.name:55s} {len(df):5d} rows | tier={df["method_tier"].unique()}')
    total_rows += len(df)
print(f'\nTotal rows across all extracted CSVs: {total_rows}')

print('\n=== Still needs extraction (in data2.xlsx) ===')
still_todo = [
    'Oddone et al. 1997 — transposed, NAA',
    'Acquafredda et al. 2018 — Italian/Sardinian sources',
    'Forster & Grave 2012 — 5 rows, oxide format',
    'ROSEN et al 2011 — transposed, microprobe oxides',
    'Carter et al. 2006 — complex 35-column layout',
    'Binder et al. 2011 — mixed oxide+ppm',
    'data2.xlsx Multiple sheet — unclear content',
]
for item in still_todo:
    print(f'  ⬜  {item}')

print('\n=== Spot-checks pending (article available) ===')
spot_checks = [
    'Carter & Shackley 2007 — Carter_and_Shackley_2007.txt ✅',
    'Carter et al. 2017 — Carter_2017_Investigating...txt ✅',
    'Khalidi et al. 2009 — khalidi Gratuze 2009.txt ✅',
    'Yellin & Perlman 1980 — Yellin_and_Perlman_1980.txt ✅',
    'Yellin & Perlman 1981 — Yellin_and_Perlman_1981.txt ✅',
    'Rosenberg & Carter 2022 — Carter_2022_Obsidian_Beads_Tel_Tsaf.txt ✅',
    'Frahm 2013 — Frahm_2013.txt ✅',
]
for item in spot_checks:
    print(f'  ⬜  {item}')